# Analysis 05 static explainability

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# Analysis 05: Static Explainability

This notebook explains how static site descriptors affect the coastal transformer heads and saves thesis-ready figures and tables under `<run_results_dir>/static_explainability/`.

Static SHAP outputs now include both grouped signed SHAP distributions, using the shared static-feature families defined elsewhere in the codebase, and conventional coloured SHAP beeswarms for the ten most important individual static features for each target. Grouped mean-absolute SHAP values are retained for ranking and cross-target summary figures.

ALE curves show how the trained model response changes across a feature range while averaging over a site-balanced held-out context. Both SHAP and ALE describe learned model relationships rather than physical causality.

For directional targets (`dir`, `dp`), signed SHAP contributions are retained from the model's internal representation, but they should not be interpreted as direct clockwise or counter-clockwise changes after circular decoding. Focus on contribution magnitude, relative ranking, and consistency across groups instead.

In [ ]:
from pathlib import Path

from notebooks.multisource_notebook_helpers import results_dir_from_config

CONFIG_PATH = "configs/FINAL_RUNS_V2/4_trans_static_cross.yaml"
RESULTS_DIR_OVERRIDE = None
RESULTS_DIR = (
    Path(RESULTS_DIR_OVERRIDE).expanduser()
    if RESULTS_DIR_OVERRIDE
    else results_dir_from_config(CONFIG_PATH)
)
TRAINING_CONFIG_PATH = CONFIG_PATH
POINT_CENTRIC_DIR = None
SPLIT = "all"  # train | val | test | all
DEVICE = "cuda"
RANDOM_SEED = 42

MAX_EXPLAIN_SAMPLES = 256
BACKGROUND_SAMPLES = 32

TARGETS_TO_ANALYZE = ["hs", "tp", "dir", "dp"]
EXPLAIN_QUANTITY = "physical_prediction"  # physical_prediction | absolute_error | signed_error | entropy | selected_logit

USE_STRATIFIED_SAMPLING = True
SAMPLING_STRATA = ["site", "target_hs_bin", "target_tp_bin"]
STATIC_GROUP_OVERRIDES = {}
STATIC_SHAP_DISPLAY_PERCENTILE_RANGE = (
    20,
    99,
)  # e.g. (1, 99) to hide rows with near-zero displayed SHAP across the top features

ALE_SITE_SET = "heldout"  # "test" | "val" | "heldout"
ALE_SAMPLES_PER_SITE = 32

ALE_FEATURE_SELECTIONS = {
    "hs": ["ray_fetch_mean_m", "ray_fetch_max_m", "local_depth_m", "open_sector_width_deg"],
    "tp": ["ray_fetch_mean_m", "ray_fetch_max_m", "local_depth_m", "open_sector_width_deg"],
    "dir": ["open_sector_width_deg", "ray_fetch_mean_m", "path_length_m", "local_depth_m"],
    "dp": ["open_sector_width_deg", "ray_fetch_max_m", "ray_fetch_mean_m", "local_depth_m"],
}
ALE_BINS = 6
ALE_MIN_BIN_SAMPLES = 10
ALE_MIN_UNIQUE_SITES_PER_BIN = 5
ALE_PERCENTILE_RANGE = (1.0, 99.0)
ALE_BOOTSTRAP_REPEATS = 500  # set to 0 to disable bootstrap confidence intervals
ALE_RANDOM_SEED = 42

SITEWISE_K_NEAREST_ANALOGUES = 5

In [ ]:
import importlib
import pandas as pd

import src.diagnostics.explainability as _explainability
import src.diagnostics as _diagnostics

importlib.reload(_explainability)
importlib.reload(_diagnostics)

from src.diagnostics import run_static_explainability_analysis

bundle = _explainability.load_results_bundle(
    RESULTS_DIR,
    training_config_path=TRAINING_CONFIG_PATH,
    point_centric_dir=POINT_CENTRIC_DIR,
)
output_dirs = _explainability._resolve_static_output_dirs(bundle)

print("Results directory:", bundle.results_dir)
print("Resolved static explainability output directory:", output_dirs["root"])
print("Split:", SPLIT)
print("Targets:", TARGETS_TO_ANALYZE)
print("Explain quantity:", EXPLAIN_QUANTITY)

In [ ]:
feature_metadata = _explainability.load_feature_metadata(bundle)
static_feature_groups = _explainability.build_feature_groups(
    feature_metadata.get("static_feature_names", []),
    overrides=STATIC_GROUP_OVERRIDES,
)

rows = []
for group_name, feature_names in static_feature_groups["groups"].items():
    for feature_name in feature_names:
        rows.append(
            {
                "group": group_name,
                "feature": feature_name,
                "matched_pattern": feature_name not in static_feature_groups["unmatched"],
            }
        )

groups_df = pd.DataFrame(rows).sort_values(["group", "feature"]).reset_index(drop=True)
groups_df.head(12)

## Interpretation Notes

- Grouped static SHAP plots show how broad static feature families contribute to the model output for the sampled cases relative to the chosen background set.
- Top-ten feature-level SHAP beeswarms show how high and low values of the most important individual static features affect each target.
- ALE shows how the model response changes across a supported feature range while averaging out the joint distribution of the other inputs.
- Neither SHAP nor ALE should be treated as proof of physical causality. They summarize the behavior of the fitted model for this run and split.

In [ ]:
outputs = run_static_explainability_analysis(
    results_dir=RESULTS_DIR,
    split=SPLIT,
    device=DEVICE,
    max_explain_samples=MAX_EXPLAIN_SAMPLES,
    background_samples=BACKGROUND_SAMPLES,
    random_seed=RANDOM_SEED,
    targets_to_analyze=TARGETS_TO_ANALYZE,
    explain_quantity=EXPLAIN_QUANTITY,
    use_stratified_sampling=USE_STRATIFIED_SAMPLING,
    sampling_strata=SAMPLING_STRATA,
    static_group_overrides=STATIC_GROUP_OVERRIDES,
    static_shap_display_percentile_range=STATIC_SHAP_DISPLAY_PERCENTILE_RANGE,
    training_config_path=TRAINING_CONFIG_PATH,
    point_centric_dir=POINT_CENTRIC_DIR,
    ale_feature_selections=ALE_FEATURE_SELECTIONS,
    ale_bins=ALE_BINS,
    ale_min_bin_samples=ALE_MIN_BIN_SAMPLES,
    ale_percentile_range=ALE_PERCENTILE_RANGE,
    ale_bootstrap_repeats=ALE_BOOTSTRAP_REPEATS,
    ale_site_set=ALE_SITE_SET,
    ale_samples_per_site=ALE_SAMPLES_PER_SITE,
    ale_min_unique_sites_per_bin=ALE_MIN_UNIQUE_SITES_PER_BIN,
    ale_random_seed=ALE_RANDOM_SEED,
    sitewise_k_nearest_analogues=SITEWISE_K_NEAREST_ANALOGUES,
)
outputs["run_summary"]

In [ ]:
sitewise_analogue = outputs.get("sitewise_static_analogue_error", pd.DataFrame())

if sitewise_analogue.empty:
    print("No site-wise analogue-distance table was generated for this run.")
else:
    display(
        sitewise_analogue[
            [
                "site",
                "split",
                "target",
                "normalized_rmse",
                "mean_training_analogue_distance",
                "nearest_training_distance",
                "sample_count",
            ]
        ]
        .sort_values(["target", "split", "mean_training_analogue_distance", "site"])
        .reset_index(drop=True)
    )
    fig = _explainability.plot_sitewise_error_vs_analog_distance_grid(
        sitewise_analogue,
        title="Site-wise normalized RMSE vs training analogue distance",
    )
    display(fig)

In [ ]:
pd.DataFrame(
    [
        {
            "field": key,
            "value": ", ".join(value) if isinstance(value, list) else value,
        }
        for key, value in outputs["artifact_summary"].items()
    ]
)

In [ ]:
outputs.get("ale_reliability_summary", pd.DataFrame())[
    [
        "target",
        "feature_label",
        "feature_range_min",
        "feature_range_max",
        "n_unique_sites",
        "reliable_bins",
        "normalized_ale_range",
        "reliability_flag",
    ]
].rename(
    columns={
        "feature_label": "analysis",
        "feature_range_min": "feature_range_min",
        "feature_range_max": "feature_range_max",
        "n_unique_sites": "unique_sites",
        "reliable_bins": "reliable_bins",
        "normalized_ale_range": "normalized_effect_range",
        "reliability_flag": "reliability",
    }
)